# FleetGuard AI — обучение YOLO11n-seg на CarDD + Carvengers (Kaggle)

**Что нужно перед запуском**

1. `Settings → Accelerator → GPU T4 x2` (одна T4 тоже сработает, но 100 эпох не успеют).
2. `Settings → Internet → On`.
3. Ключ Roboflow: `Add-ons → Secrets` → добавить секрет `ROBOFLOW_API_KEY`
   (ключ: `universe.roboflow.com` → `Settings` → `API Keys`). Без секрета ячейка спросит ключ вручную.
4. `Run → Run all`.

**Что делает ноутбук:** клонирует репозиторий, скачивает CarDD (4000 изображений, полигоны) и
Carvengers (17293 изображения, боксы), приводит классы к одной схеме, конвертирует боксы в полигоны,
объединяет датасеты, дублирует `crack` в train, обучает 100 эпох и снимает метрики `model.val()`.

**Сколько ждать:** 20–40 минут на данные, затем ~6–7 часов на 100 эпох на двух T4
(на одной T4 — ~11–12 часов, это упирается в лимит сессии Kaggle).

**Почему Kaggle, а не Colab:** сессия Kaggle живёт 12 часов и даёт 30 GPU-часов в неделю, поэтому
100 эпох закрываются за один прогон. Каждая эпоха печатает строку с ETA, обучение можно прервать и
продолжить с `last.pt`.

**В конце** ноутбук напечатает `metrics.md`, `sha256` и путь к `best.pt`.

## 1. Проверка окружения

In [ ]:
import subprocess
import sys
import torch

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpus:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(" ", index, torch.cuda.get_device_name(index))
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    or "GPU не найден — включите Settings → Accelerator → GPU T4 x2"
)

## 2. Репозиторий и зависимости

`USE_BOTH_GPUS = True` включает DDP на двух T4: эпоха идёт примерно вдвое быстрее.
Если DDP падает с ошибкой NCCL, поставьте `False` и перезапустите — тогда 100 эпох
не влезут в одну сессию, но прогресс сохраняется и обучение можно продолжить.

In [ ]:
import subprocess
import sys
from pathlib import Path

USE_BOTH_GPUS = True

REPO = Path("/kaggle/working/fleetguard-ai")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/minbaevv/fleetguard-ai.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-input",
        "ultralytics>=8.3,<9.0",
        "huggingface_hub",
        "roboflow",
    ],
    check=True,
)
import ultralytics

print("repo:", REPO)
print("ultralytics:", ultralytics.__version__)
print("DDP:", "2 GPU" if USE_BOTH_GPUS else "1 GPU")

## 3. CarDD: скачивание

Публичное зеркало `shawnmichael/CarDD` (4000 изображений: 2816 train / 810 val / 374 test).
Лицензия набора — исследовательская/некоммерческая.

In [ ]:
import os
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download

REPO = Path("/kaggle/working/fleetguard-ai")
DATA = REPO / "datasets" / "cardd"

snapshot_download(
    repo_id="shawnmichael/CarDD",
    repo_type="dataset",
    local_dir=str(DATA),
    allow_patterns=[
        "images/train/*.jpg",
        "images/val/*.jpg",
        "images/test/*.jpg",
        "annotations/instances_*.json",
    ],
    max_workers=8,
)

images = DATA / "images_all"
images.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    for image in (DATA / "images" / split).glob("*.jpg"):
        target = images / image.name
        if not target.exists():
            try:
                os.link(image, target)
            except OSError:
                shutil.copy2(image, target)
print("images:", len(list(images.glob("*.jpg"))))

## 4. CarDD → YOLO-seg

`merge_coco_splits.py` собирает три COCO-файла в один, `prepare_dataset.py` режет 70/15/15
с seed 42 и пишет нормализованные полигоны. Тестовый сплит остаётся нетронутым.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/fleetguard-ai")
DATA = REPO / "datasets" / "cardd"
annotations = [
    str(DATA / "annotations" / f"instances_{split}2017.json")
    for split in ("train", "val", "test")
]
merged = DATA / "instances_all.json"

subprocess.run(
    [sys.executable, "cv/training/merge_coco_splits.py", "--annotations", *annotations, "--out", str(merged)],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "cv/training/prepare_dataset.py",
        "--images",
        str(DATA / "images_all"),
        "--annotations",
        str(merged),
        "--out",
        str(REPO / "datasets" / "cardd-yolo"),
        "--seed",
        "42",
        "--force",
    ],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [sys.executable, "cv/training/train.py", "--data", str(REPO / "datasets" / "cardd-yolo"), "--dry-run"],
    cwd=REPO,
    check=True,
)

## 5. Carvengers → слияние с CarDD

Carvengers — object detection (17293 изображения, CC BY 4.0). Боксы становятся прямоугольными
полигонами, поэтому маски для этих изображений грубее, чем в CarDD. Точные дубликаты удаляются по
sha256, `crack` дублируется только в train.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO = Path("/kaggle/working/fleetguard-ai")
RAW = REPO / "datasets" / "carvengers"

if not list(RAW.rglob("data.yaml")):
    api_key = os.environ.get("ROBOFLOW_API_KEY", "")
    if not api_key:
        try:
            from kaggle_secrets import UserSecretClient

            api_key = UserSecretClient().get_secret("ROBOFLOW_API_KEY") or ""
        except Exception:
            api_key = ""
    if not api_key:
        api_key = getpass(
            "Roboflow API key (universe.roboflow.com -> Settings -> API Keys): "
        ).strip()
    if not api_key:
        raise SystemExit("Без ключа Roboflow внешний датасет не скачать")
    from roboflow import Roboflow

    project = (
        Roboflow(api_key=api_key)
        .workspace("carvengers")
        .project("carvengers-damage-detection")
    )
    project.version(1).download("yolov8", location=str(RAW), overwrite=True)
    print("roboflow: загружено")

extra = next(iter(RAW.rglob("data.yaml"))).parent
print("extra:", extra)

subprocess.run(
    [
        sys.executable,
        "cv/training/merge_yolo_datasets.py",
        "--base",
        str(REPO / "datasets" / "cardd-yolo"),
        "--extra",
        str(extra),
        "--out",
        str(REPO / "datasets" / "cardd-carvengers-yolo"),
        "--oversample-class",
        "crack",
        "--oversample-factor",
        "2",
        "--force",
    ],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "cv/training/train.py",
        "--data",
        str(REPO / "datasets" / "cardd-carvengers-yolo"),
        "--dry-run",
    ],
    cwd=REPO,
    check=True,
)

## 6. Baseline на том же merged val (необязательно)

Чтобы сравнение «до/после» было честным, старая модель (`best.pt` прогона на CarDD) оценивается
на объединённом val. Загрузите этот файл как Kaggle Dataset (`Add Input → Upload`) — ячейка найдёт
его в `/kaggle/input`. Если файла нет, ячейка просто скажет об этом и продолжит.

In [ ]:
import sys
from pathlib import Path

from ultralytics import YOLO

REPO = Path("/kaggle/working/fleetguard-ai")
sys.path.insert(0, str(REPO / "cv" / "training"))

from train import collect_metrics, validate_dataset

RUNS = REPO / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
DATA = REPO / "datasets" / "cardd-carvengers-yolo"

candidates = sorted(Path("/kaggle/input").rglob("best.pt"))
baseline = candidates[0] if candidates else None
summary = validate_dataset(DATA)

if baseline is None:
    print("baseline best.pt не найден в /kaggle/input — сравнение до/после будет не с чем")
    print("загрузите его как Kaggle Dataset и перезапустите ячейку")
else:
    print("baseline:", baseline)
    metrics = collect_metrics(YOLO(str(baseline)), summary.yaml_path, "val", summary.names)
    print("baseline on merged val")
    print(metrics["markdown"])
    (RUNS / "baseline-on-merged-val.md").write_text(
        metrics["markdown"] + "\n", encoding="utf-8"
    )

## 7. Обучение 100 эпох

Если в `runs/yolo11n-seg-cardd-carvengers/weights/last.pt` уже есть чекпоинт **этого** прогона,
обучение продолжается с него, а не начинается заново.

Прогресс печатается каждые 30 секунд по `results.csv`: номер эпохи, минуты на эпоху, ETA и
`mask mAP50-95`. Так сделано потому, что под DDP Ultralytics запускает обучение в дочерних
процессах и колбэки родителя там не работают — единственный надёжный источник прогресса это
файл `results.csv`.

`TIME_BUDGET_H` — страховка от обрыва сессии. Когда бюджет исчерпан, ноутбук посылает `SIGINT`
всей группе процессов, обучение закрывает текущую эпоху и сохраняет `best.pt`/`last.pt`.

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/kaggle/working/fleetguard-ai")
RUNS = REPO / "runs"
NAME = "yolo11n-seg-cardd-carvengers"
DATA = REPO / "datasets" / "cardd-carvengers-yolo"
TOTAL_EPOCHS = 100
TIME_BUDGET_H = 10.0  # страховка от обрыва сессии; при 2×T4 100 эпох идут ~6–7 ч

results_csv = RUNS / NAME / "results.csv"
last_pt = RUNS / NAME / "weights" / "last.pt"
args_yaml = RUNS / NAME / "args.yaml"

resume = False
if last_pt.is_file():
    # Ultralytics восстанавливает save_dir из чекпоинта, поэтому прогон, начатый на
    # другой машине (Colab: /content/drive/...), продолжил бы писать туда. Возобновляем
    # только прогоны, чей args.yaml указывает на этот каталог Kaggle.
    recorded = args_yaml.read_text(encoding="utf-8") if args_yaml.is_file() else ""
    if str(RUNS) in recorded:
        resume = True
        print("resume: продолжаем с", last_pt)
    else:
        print("найден чужой чекпоинт, он не переиспользуется:", last_pt)
        print("args.yaml указывает на другой каталог — начинаем чистый прогон")

device = "0,1" if USE_BOTH_GPUS else "0"
# batch остаётся 16: при DDP Ultralytics делит его между GPU, эффективный размер
# батча совпадает с прогоном на одной T4, поэтому метрики сравнимы.
batch = 16

command = [
    sys.executable,
    "cv/training/train.py",
    "--data",
    str(DATA),
    "--device",
    device,
    "--batch",
    str(batch),
    "--project",
    str(RUNS),
    "--name",
    NAME,
    "--time-hours",
    str(TIME_BUDGET_H),
]
if resume:
    command.append("--resume")
print(" ".join(command))


def stop_group(proc: subprocess.Popen, grace: int = 180) -> None:
    # SIGINT всей группе: DDP-воркеры успевают дописать эпоху и сохранить веса.
    try:
        group = os.getpgid(proc.pid)
    except ProcessLookupError:
        return
    os.killpg(group, signal.SIGINT)
    try:
        proc.wait(timeout=grace)
    except subprocess.TimeoutExpired:
        os.killpg(group, signal.SIGKILL)


started = time.monotonic()
proc = subprocess.Popen(command, cwd=REPO, start_new_session=True)
seen = 0
while proc.poll() is None:
    time.sleep(30)
    if time.monotonic() - started > TIME_BUDGET_H * 3600:
        print(f"бюджет {TIME_BUDGET_H:g} ч исчерпан — останавливаю обучение на текущей эпохе")
        stop_group(proc)
        break
    if not results_csv.is_file():
        continue
    rows = results_csv.read_text(encoding="utf-8").strip().splitlines()
    done = len(rows) - 1
    if done <= seen or done <= 0:
        continue
    header = [cell.strip() for cell in rows[0].split(",")]
    record = dict(zip(header, rows[-1].split(",")))
    per_epoch = float(record["time"]) / done
    eta = per_epoch * max(TOTAL_EPOCHS - done, 0)
    mask = record.get("metrics/mAP50-95(M)", "?")
    print(
        f"epoch {done}/{TOTAL_EPOCHS} ({done * 100 // TOTAL_EPOCHS}%)"
        f" | {per_epoch / 60:.1f} min/epoch"
        f" | ETA {eta / 3600:.1f} h"
        f" | mask mAP50-95 {mask}",
        flush=True,
    )
    seen = done

print("exit code:", proc.returncode)

## 8. Метрики и артефакты

Считаются на `val` объединённого датасета по `best.pt`. Тестовый сплит не используется.

In [ ]:
import hashlib
import sys
from pathlib import Path

from ultralytics import YOLO

REPO = Path("/kaggle/working/fleetguard-ai")
sys.path.insert(0, str(REPO / "cv" / "training"))

from train import collect_metrics, completed_epochs, validate_dataset, write_metrics

RUNS = REPO / "runs"
NAME = "yolo11n-seg-cardd-carvengers"
run_dir = RUNS / NAME
best = run_dir / "weights" / "best.pt"
DATA = REPO / "datasets" / "cardd-carvengers-yolo"
PLANNED_EPOCHS = 100

if not best.is_file():
    raise SystemExit(f"best.pt не найден: {best}")

summary = validate_dataset(DATA)
metrics = collect_metrics(YOLO(str(best)), summary.yaml_path, "val", summary.names)
finished = completed_epochs(run_dir)

report = {
    "model": "yolo11n-seg.pt",
    "dataset": str(summary.yaml_path),
    "epochs": PLANNED_EPOCHS,
    "epochs_completed": finished,
    "seed": 42,
    "split": metrics["split"],
    "overall": metrics["overall"],
    "classes": metrics["classes"],
    "weights": str(best),
    "weights_sha256": hashlib.sha256(best.read_bytes()).hexdigest(),
    "resumed_from": None,
    "stopped_early": finished is not None and finished < PLANNED_EPOCHS,
}
write_metrics(run_dir, report, metrics["markdown"])

print("best.pt:", best, "| MB:", round(best.stat().st_size / 1024 / 1024, 1))
print("sha256:", report["weights_sha256"])
print()
print(metrics["markdown"])

baseline = RUNS / "baseline-on-merged-val.md"
print()
print("baseline на merged val:")
print(baseline.read_text(encoding="utf-8") if baseline.is_file() else "не считался")

print()
print("закрыто эпох:", finished if finished is not None else "неизвестно", "из", PLANNED_EPOCHS)
print("артефакты:", run_dir)

## Что прислать в чат

1. Таблицу метрик из последней ячейки и, если он есть, baseline на merged val.
2. Строку `sha256`.
3. Ссылку на `best.pt`: правая панель `Output` → `runs/yolo11n-seg-cardd-carvengers/weights/best.pt`
   → `Download`, затем загрузить на Google Drive и дать ссылку «все, у кого есть ссылка».

Веса в git не коммитятся. В README публикуются только измеренные значения.